# Score `patch_s2` on the GIFT-Eval test split (once)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shmorsi/tsfm-from-scratch/blob/main/notebooks/score_patch_s2_test.ipynb)

**Time, before Run all.** A free Colab T4 can disconnect. Each finished test configuration
is written to the Drive `WORK` folder, and Run all again continues from those files. A
completed `all_results.csv` is not scored again.

No test-split duration is stored. `results/bench/Tesla_T4.json` measured 10
steps and projects **1.29 hours** to train `patch_s2` (12,000 configured
steps, 0.386 s/step, peak 1.52 GB, fp16). That is a training
projection, not a test-split scoring time. The validation score of this checkpoint
stored a `seconds` field on each configuration in `results/eval/val/patch_s2/run.json`.
Those values are not a forecast of the test split.

`patch_s2` (10,083,392 parameters) was chosen on the validation split by the rule in
`PREREGISTRATION.md` section 4. Its checkpoint stayed on Google Drive, so it has no
test score yet. This notebook gives it the pre-registered one, on a Colab T4. Do not
paste in outputs from another machine. The parameter count and the checkpoint hash
below are read from the committed validation run when this notebook is generated.

The weights file recorded with that run has SHA-256 `85c46256cefa665b7eb0822f9877831563df85e22345670b40c4a115c16d0fd2`.
`ckpt.pt` or the sibling `model.pt` both work; the script scores the file with that hash.

**Before Run all**

1. Merge the commit you want scored, or set `COMMIT` below to that branch. The notebook
   clones `COMMIT` (default `main`) and scores that revision.
2. Runtime -> Change runtime type -> T4 GPU.
3. Leave `CHECKPOINT_PATH` when the run folder is already at the default Drive path.
   It has to be that `patch_s2` run, not a neighbour such as `bins_s2`.

**What the pre-registration fixes, and this notebook follows**

* Test split: the 97 official configurations, at the pinned revision (section 2).
* Metrics: GluonTS MASE and CRPS over Seasonal Naive, geometric mean (section 3).
* Inference: context up to 2,048 points, patch roll-out by feeding back medians (section 5).
* Decision rule: CRPS against upstream AutoARIMA, 95% paired bootstrap over
  configurations; better, worse or no detectable difference (section 5).
* Scored once (section 6). `tsfm eval` refuses a second test score, and refuses any
  model test score unless the `prereg-v1` tag is in the commit's history.

**What this run is not.** It is one seed. `DEVIATIONS.md` (2026-10-15) already records
that seeds 1 and 2 were not trained, so the interval covers configurations, not seeds.
It is not a second look: the 2026-10-15 attempt disconnected partway and its partial
outputs were never inspected. This notebook writes to a new folder and never reads them.
Do not use `notebooks/colab_runner.ipynb` for this score. That queue can resume those
partial outputs, and it would also start the volatility-model stages that the same file
records as not run.
`PREREGISTRATION_CPU.md` does not authorise or forbid this run; the CPU study is a
separate model with its own test score.

**After it finishes, on a local clone of the same commit.** Download
`harvest_patch_s2.txt` from the Drive `WORK` folder. A copied notebook print can
truncate, and a truncated file fails the SHA-256 check. Then:

    uv run python scripts/harvest_colab_output.py harvest_patch_s2.txt
    uv run tsfm docs build

Commit the harvested result files and the docs that command regenerates. Do not
hand-edit numbers. The harvest file is what the doc generators read:

* `results/eval/test/patch_s2/all_results.csv`, `run.json` and `config.json`
* `results/analysis/test_summary.json` (rebuilt by `tsfm report`; this replaces the previous file)
* `results/analysis/headline.json` (new)

`scripts/harvest_colab_output.py` also appends `results/sync_log.json`.
`results/analysis/cpu_study.json` is not in the harvest. Ranks inside the rebuilt
`test_summary.json` include `patch_s2`. CRPS and MASE for the other models are
recomputed from the same vendored files.

Re-run the notebook after a disconnect. Finished configurations stay on Drive, and the
script does not score the test split a second time once `all_results.csv` is complete.


In [ ]:
# Parameters
# ============================================================================
print(
    "results/bench/Tesla_T4.json training projection for patch_s2: "
    "1.29 hours (12,000 configured steps, 0.386 s/step, "
    "10 timed steps, peak 1.52 GB, fp16). "
    "Not a test-split scoring time. No test-split duration is stored. "
    "Finished configurations resume from WORK on Drive."
)
# CHECKPOINT_PATH: the patch_s2 run folder on Drive. Set COMMIT as well when the
# revision to score is not on main yet.
# ckpt.pt (training checkpoint) or model.pt (inference weights) both work: the
# script checks the run config and the parameter count, then scores the file
# whose SHA-256 matches the validation run (printed in the cell above).
CHECKPOINT_PATH = "/content/drive/MyDrive/tsfm/runs/patch_s2/ckpt.pt"
# ============================================================================
REPO = "shmorsi/tsfm-from-scratch"
COMMIT = "main"  # or a full commit SHA, or a branch name
WORK = "/content/drive/MyDrive/tsfm/score_patch_s2"  # resumable outputs

In [ ]:
# 1. Mount Drive (checkpoint in, resumable results out)
import os

from google.colab import drive

drive.mount("/content/drive")
assert os.path.isfile(CHECKPOINT_PATH), f"CHECKPOINT_PATH not found: {CHECKPOINT_PATH}"
os.makedirs(WORK, exist_ok=True)

In [ ]:
# 2. Clone the repo at COMMIT, with tags (tsfm eval checks for prereg-v1)
import os
import subprocess

if not os.path.exists("/content/tsfm"):
    subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git", "/content/tsfm"], check=True)
subprocess.run(["git", "-C", "/content/tsfm", "fetch", "-q", "--force", "--tags", "origin"], check=True)
remote = f"origin/{COMMIT}"
known = subprocess.run(["git", "-C", "/content/tsfm", "rev-parse", "-q", "--verify", remote], capture_output=True)
ref = remote if known.returncode == 0 else COMMIT
subprocess.run(["git", "-C", "/content/tsfm", "checkout", "-q", "--force", "--detach", ref], check=True)
print(subprocess.run(["git", "-C", "/content/tsfm", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# 3. Install (uv, locked dependencies, CUDA 12.6 PyTorch build)
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null
import os

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["TSFM_HOME"] = "/content/tsfm_home"
%cd /content/tsfm
!uv sync --locked --extra cu126 -q


In [ ]:
# 4. GPU check with the PyTorch this repo pins (not Colab's preinstalled build)
import subprocess

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(smi.stdout)
probe = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-c",
        "import torch; name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''; print(torch.__version__); print(torch.cuda.is_available()); print(name)",
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(probe.stdout)
lines = probe.stdout.splitlines()
assert len(lines) >= 3, "GPU probe did not print version, availability and device name"
assert lines[1] == "True", "No GPU: Runtime -> Change runtime type -> T4 GPU"
assert "T4" in lines[2], "Need a T4 GPU: Runtime -> Change runtime type -> T4 GPU. Got: " + lines[2]

In [ ]:
# 5. Check the checkpoint is patch_s2 before anything touches the test split
!uv run python scripts/score_patch_s2.py --checkpoint "{CHECKPOINT_PATH}" --work "{WORK}" --dry-run

In [ ]:
# 6. Score once: GIFT-Eval download (SHA-256 checked), tsfm eval --split test,
#    tsfm report, tsfm headline, tsfm docs build. Re-run after a disconnect.
!uv run python scripts/score_patch_s2.py --checkpoint "{CHECKPOINT_PATH}" --work "{WORK}" --device cuda

In [ ]:
# 7. Harvest file on Drive. Download WORK/harvest_patch_s2.txt (do not copy a
#    truncated print). On a local clone of this COMMIT:
#    uv run python scripts/harvest_colab_output.py harvest_patch_s2.txt
#    uv run tsfm docs build
print("download this file from Drive:", f"{WORK}/harvest_patch_s2.txt")
print(open(f"{WORK}/harvest_patch_s2.txt").read())
!git -C /content/tsfm status --short